# listing-extract-llm — Walkthrough

Fine-tuning a small open-source LLM to pull structured attribute-value JSON out of raw marketplace listings (title + description), using the [WDC PAVE](https://webdatacommons.org/structureddata/wdc-pave/) benchmark.

**Sections:**
1. Setup
2. Config
3. Load & explore data
4. Define the extraction schema & prompt
5. Baseline eval (before fine-tuning)
6. Fine-tuning (LoRA/SFT — run on Colab GPU)
7. Post-fine-tuning eval
8. Before/after comparison
9. Qualitative demo
10. Serving (pointer to `src/serve.py`)

## 1. Setup

In [1]:
import json
import sys
from pathlib import Path

import torch
from dotenv import load_dotenv

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT / "src"))
load_dotenv(PROJECT_ROOT / ".env")

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"torch {torch.__version__}, device: {device}")

torch 2.14.0+cpu, device: cpu


## 2. Config

Scoping to a single category first (`Computers And Accessories` which is the largest slice) keeps the attribute schema small and training fast.

In [2]:
CONFIG = {
    "base_model": "Qwen/Qwen2.5-0.5B-Instruct",
    "category": "Computers And Accessories",
    "data_dir": PROJECT_ROOT / "data",
    "eval_sample_size": 60,
    "seed": 42,
    "max_seq_length": 512,
    "finetuned_model_id": None,  # e.g. "your-hf-username/qwen2.5-0.5b-listing-extract" once pushed
}
CONFIG

{'base_model': 'Qwen/Qwen2.5-0.5B-Instruct',
 'category': 'Computers And Accessories',
 'data_dir': WindowsPath('c:/Local Programming Projects/listing-extract-llm/data'),
 'eval_sample_size': 60,
 'seed': 42,
 'max_seq_length': 512,
 'finetuned_model_id': None}

## 3. Load & explore data



In [3]:
def load_split(name: str) -> list[dict]:
    path = CONFIG["data_dir"] / f"{name}.jsonl"
    with path.open(encoding="utf-8") as f:
        return [json.loads(line) for line in f]


train_rows = load_split("train")
val_rows = load_split("val")
test_rows = load_split("test")

train_cat = [r for r in train_rows if r["category"] == CONFIG["category"]]
val_cat = [r for r in val_rows if r["category"] == CONFIG["category"]]
test_cat = [r for r in test_rows if r["category"] == CONFIG["category"]]

print(f"{CONFIG['category']}: {len(train_cat)} train / {len(val_cat)} val / {len(test_cat)} test")
train_cat[0]

Computers And Accessories: 305 train / 44 val / 87 test


{'id': 13803205,
 'category': 'Computers And Accessories',
 'input_title': '653957-001 HP G8 G9 600-GB 6G 10K 2.5 SAS SC',
 'input_description': 'Description:HP 600GB 2.5-inch SFF Serial Attached SCSI (SAS)6G 10K Enterprise Hot-Plug Hard Drive in G8 G9(Gen8 Gen9) SmartDrive Carrier (SC) (as pictured)Genuine HPE serial number and firmwareGenuine Certified DrivePart Number(s) Option Part# 652583-B21 Spare 653957-001 Assembly 507129-014 641552-003 666355-003 599476-003 Model# EA1233000BU EG0600FBVFP EG0600FBDSR EG0600FCHHU SmartBuy 652583-S21"',
 'gold_json': {'Generation': 'Generation 8 Generation 9',
  'Part Number': '653957001',
  'Product Type': 'Storage Solutions',
  'Cache': None,
  'Processor Type': None,
  'Processor Core': None,
  'Interface': 'SAS (Serial Attached SCSI) Interfaces',
  'Manufacturer': ['Hewlett-Packard', 'Hewlett-Packard Enterprise'],
  'Capacity': '600 Gigabytes',
  'Ports': None,
  'Rotational Speed': '10000'}}

In [4]:
def get_attributes(rows: list[dict]) -> list[str]:
    attrs: set[str] = set()
    for r in rows:
        attrs.update(r["gold_json"].keys())
    return sorted(attrs)


ATTRIBUTES = get_attributes(train_cat)
print(f"{len(ATTRIBUTES)} attributes: {ATTRIBUTES}")

11 attributes: ['Cache', 'Capacity', 'Generation', 'Interface', 'Manufacturer', 'Part Number', 'Ports', 'Processor Core', 'Processor Type', 'Product Type', 'Rotational Speed']


In [5]:
SYSTEM_PROMPT = (
    "You are a product-listing extraction assistant. Given a listing title and description, "
    "extract values for the following attributes: " + ", ".join(ATTRIBUTES) + ". "
    "Respond with a single JSON object using exactly these keys. "
    "Use null for any attribute that isn't mentioned in the text."
)


def build_messages(title: str, description: str) -> list[dict]:
    user_content = f"Title: {title}\nDescription: {description}"
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_content},
    ]


example = train_cat[0]
build_messages(example["input_title"], example["input_description"])

[{'role': 'system',
  'content': "You are a product-listing extraction assistant. Given a listing title and description, extract values for the following attributes: Cache, Capacity, Generation, Interface, Manufacturer, Part Number, Ports, Processor Core, Processor Type, Product Type, Rotational Speed. Respond with a single JSON object using exactly these keys. Use null for any attribute that isn't mentioned in the text."},
 {'role': 'user',
  'content': 'Title: 653957-001 HP G8 G9 600-GB 6G 10K 2.5 SAS SC\nDescription: Description:HP 600GB 2.5-inch SFF Serial Attached SCSI (SAS)6G 10K Enterprise Hot-Plug Hard Drive in G8 G9(Gen8 Gen9) SmartDrive Carrier (SC) (as pictured)Genuine HPE serial number and firmwareGenuine Certified DrivePart Number(s) Option Part# 652583-B21 Spare 653957-001 Assembly 507129-014 641552-003 666355-003 599476-003 Model# EA1233000BU EG0600FBVFP EG0600FBDSR EG0600FCHHU SmartBuy 652583-S21"'}]

## 5. Baseline eval — before fine-tuning

Same idea as the reference video: run the untouched base model on a held-out sample and see how bad it is at following the schema. This is the "before" column in the results table.

In [6]:
from transformers import AutoModelForCausalLM, AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(CONFIG["base_model"])
base_model = AutoModelForCausalLM.from_pretrained(CONFIG["base_model"], torch_dtype="auto").to(device)

c:\Local Programming Projects\listing-extract-llm\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Local Programming Projects\listing-extract-llm\.venv\Lib\site-packages\huggingface_hub\file_download.py:150: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\juans\.cache\huggingface\hub\models--Qwen--Qwen2.5-0.5B-Instruct. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python a

In [7]:
def generate_json(model, tokenizer, title: str, description: str, max_new_tokens: int = 300) -> str:
    messages = build_messages(title, description)
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    output = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    generated = output[0][inputs["input_ids"].shape[1] :]
    return tokenizer.decode(generated, skip_special_tokens=True)

In [ ]:
def try_parse_json(text: str) -> dict | None:
    start, end = text.find("{"), text.rfind("}")
    if start == -1 or end == -1:
        return None
    try:
        return json.loads(text[start : end + 1])
    except json.JSONDecodeError:
        return None


def evaluate(model, tokenizer, rows: list[dict]) -> dict:
    valid_json = 0
    schema_ok = 0
    field_correct = 0
    field_total = 0

    for row in rows:
        raw = generate_json(model, tokenizer, row["input_title"], row["input_description"])
        parsed = try_parse_json(raw)
        gold = row["gold_json"]

        if parsed is not None:
            valid_json += 1
            if set(parsed.keys()) == set(ATTRIBUTES):
                schema_ok += 1
            for attr in ATTRIBUTES:
                field_total += 1
                if parsed.get(attr) == gold.get(attr):
                    field_correct += 1
        else:
            field_total += len(ATTRIBUTES)

    n = len(rows)
    return {
        "n": n,
        "valid_json_rate": valid_json / n,
        "schema_pass_rate": schema_ok / n,
        "field_accuracy": field_correct / field_total if field_total else 0.0,
    }

In [9]:
import random

random.seed(CONFIG["seed"])
eval_rows = random.sample(test_cat, min(CONFIG["eval_sample_size"], len(test_cat)))

baseline_metrics = evaluate(base_model, tokenizer, eval_rows)
baseline_metrics

{'n': 60,
 'valid_json_rate': 1.0,
 'schema_pass_rate': 0.7666666666666667,
 'field_accuracy': 0.11969696969696969}

## 6. Fine-tuning (LoRA / SFT — run on a GPU)

The LoRA config and training setup:
- Format `train_cat`/`val_cat` into chat-style examples using `build_messages` + the target `gold_json` as the assistant turn
- `peft.LoraConfig` (rank ~8-16, target the attention projection modules)
- `trl.SFTTrainer` with `max_seq_length=CONFIG["max_seq_length"]`

In [10]:
# TODO: once trained (locally or on Colab), point this at your adapter
FINETUNED_ADAPTER_PATH = None  # e.g. PROJECT_ROOT / "models" / "qwen2.5-0.5b-listing-extract-lora"

## 7. Post-fine-tuning eval

Load the base model + LoRA adapter, run the exact same `evaluate()` on the exact same `eval_rows`, and compare directly against `baseline_metrics`.

In [11]:
# TODO (after section 6 is done):
# from peft import PeftModel
# ft_model = PeftModel.from_pretrained(base_model, FINETUNED_ADAPTER_PATH).to(device)
# finetuned_metrics = evaluate(ft_model, tokenizer, eval_rows)
# finetuned_metrics

## 8. Before/after comparison

This table (plus the paper's published GPT-3.5/GPT-4 F1 numbers as a reference point) is the headline result for the README.

In [12]:
# TODO (after section 7):
# import pandas as pd
# pd.DataFrame([
#     {"stage": "base", **baseline_metrics},
#     {"stage": "fine-tuned", **finetuned_metrics},
# ])

## 9. Qualitative demo

Pick one listing and show the base model's output next to the fine-tuned model's output, side by side against the gold JSON — this is the concrete, readable proof that fine-tuning did something.

In [13]:
demo_row = eval_rows[0]
print("TITLE:", demo_row["input_title"])
print("DESCRIPTION:", demo_row["input_description"])
print("\nGOLD:", json.dumps(demo_row["gold_json"], indent=2))
print("\nBASE MODEL OUTPUT:")
print(generate_json(base_model, tokenizer, demo_row["input_title"], demo_row["input_description"]))

# TODO (after section 6): also print generate_json(ft_model, tokenizer, ...) here for the side-by-side

TITLE: HP TR-S23X-C 160-GB 230-GB LVD SE Ldr
DESCRIPTION: Description: 160GB/230GB LVD/SE Ldr Rdy SDLT Option Part# TR-S23X-C

GOLD: {
  "Generation": null,
  "Part Number": "TRS23XC",
  "Product Type": "Media and Accessories",
  "Cache": null,
  "Processor Type": null,
  "Processor Core": null,
  "Interface": "Other Interfaces",
  "Manufacturer": "Hewlett-Packard",
  "Capacity": [
    "160 Gigabytes 230 Gigabytes",
    "160 Gigabytes/230 Gigabytes"
  ],
  "Ports": null,
  "Rotational Speed": null
}

BASE MODEL OUTPUT:
{
  "Cache": "160",
  "Capacity": "230",
  "Generation": "LVD",
  "Interface": "SDLT",
  "Manufacturer": "HP",
  "Part Number": "TR-S23X-C",
  "Ports": "None",
  "Processor Core": null,
  "Processor Type": "Unknown",
  "Product Type": "Storage Device",
  "Rotational Speed": "Not Specified"
}


## 10. Serving

Once the adapter is trained, `src/serve.py` wraps it in a FastAPI endpoint (`POST /extract`, `{title, description} -> structured JSON`). Run it with:

```bash
uv run uvicorn src.serve:app --reload
```

and hit it with a quick request to confirm it behaves the same as the in-notebook `generate_json` calls above.